# GLADIUS applied workflow

This notebook fits the public, paper-reference GLADIUS path on a simulated fleet-maintenance panel. It inspects train and holdout behavior, checks identification diagnostics, reports trajectory-bootstrap intervals, re-solves a replacement-cost counterfactual with supplied planning transitions, and verifies serialization. The transition tensor is stored for planning; GLADIUS does not use it to estimate the reward.

In [ ]:
import pickle
import warnings
from pathlib import Path

import jax.numpy as jnp
import numpy as np
import pandas as pd

import econirl
from econirl import GLADIUS
from econirl.core.reward_spec import RewardSpec

checkout_root = Path.cwd().resolve().parents[1]
module_outside_checkout = not Path(econirl.__file__).resolve().is_relative_to(checkout_root)
print(f'Installed package import: {module_outside_checkout}')
print(f'Package version: {econirl.__version__}')
print(f'Package module: {econirl.__file__}')

## Simulate fleet-maintenance panels

Condition runs from 0 (healthy) to 5 (poor). Action 0 keeps the engine in service and condition deteriorates; action 1 replaces it and resets condition. Replacement has a known flow cost of 2, making it the anchor. Separate asset trajectories form the train and holdout sets.

In [ ]:
n_states, n_actions, discount = 6, 2, 0.90
transitions = np.zeros((n_actions, n_states, n_states))
for state in range(n_states):
    transitions[0, state, min(state + 1, n_states - 1)] = 1.0
    transitions[1, state, 0] = 1.0

feature_matrix = np.zeros((n_states, n_actions, 2), dtype=np.float32)
feature_matrix[:, 0, 0] = -np.arange(n_states)
feature_matrix[:, 1, 1] = -1.0
features = RewardSpec(
    jnp.asarray(feature_matrix), names=['condition_cost', 'replacement_cost']
)
true_reward = np.einsum('sak,k->sa', feature_matrix, np.array([0.4, 2.0]))
value = np.zeros(n_states)
for _ in range(1000):
    q_true = true_reward + discount * np.einsum('asn,n->sa', transitions, value)
    peak = q_true.max(axis=1)
    updated = peak + np.log(np.exp(q_true - peak[:, None]).sum(axis=1))
    if np.max(np.abs(updated - value)) < 1e-12:
        break
    value = updated
true_policy = np.exp(q_true - updated[:, None])

def simulate_assets(seed, n_assets):
    rng = np.random.default_rng(seed)
    rows = []
    for asset in range(n_assets):
        state = 0
        for period in range(20):
            action = int(rng.choice(n_actions, p=true_policy[state]))
            next_state = int(rng.choice(n_states, p=transitions[action, state]))
            rows.append({'id': asset, 'period': period, 'state': state,
                         'action': action, 'next_state': next_state})
            state = next_state
    return pd.DataFrame(rows)

train = simulate_assets(7_001, 80)
held_out = simulate_assets(7_002, 30)
print(train.groupby(['state', 'action']).size().unstack(fill_value=0))
print(f'Train rows: {len(train)}; held-out rows: {len(held_out)}')

In [ ]:
model = GLADIUS(
    n_actions=n_actions, discount=discount,
    q_hidden_dim=16, q_num_layers=1,
    ev_hidden_dim=16, ev_num_layers=1,
    batch_size=64, max_epochs=80, patience=81,
    anchor_action=1, anchor_rewards=tuple(true_reward[:, 1]),
    compute_se=True, n_bootstrap=3,
    seed=7, se_seed=8,
)
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    warnings.simplefilter('ignore', UserWarning)
    model.fit(
        train, state='state', action='action', id='id',
        features=features, transitions=transitions,
    )

assert model.objective_ == 'paper_minimax'
assert np.isfinite(model.reward_).all()
assert np.isfinite(model.policy_).all()
model.diagnostics_

In [ ]:
print(model.summary())
print('optimization:', model.diagnostics_['optimization'])
print('bootstrap draws:', model.bootstrap_.n_successful, '/', model.bootstrap_.n_requested)
intervals = model.conf_int()
print('first reward interval:', next(iter(intervals.items())))
held_out_policy = model.predict_proba(held_out['state'].to_numpy())
held_out_actions = held_out['action'].to_numpy()
held_out_nll = -np.mean(np.log(held_out_policy[np.arange(len(held_out)), held_out_actions]))
uniform_nll = np.log(n_actions)
print(f'Held-out negative log likelihood: {held_out_nll:.4f} (uniform {uniform_nll:.4f})')
assert held_out_nll < uniform_nll

## Counterfactual and serialization

A 0.5 reduction in replacement cost is structurally supported because the fit supplied both the known replacement anchor and a baseline transition tensor for planning.

In [ ]:
reward_delta = np.zeros_like(model.reward_)
reward_delta[:, 1] = 0.5
counterfactual = model.counterfactual(
    reward_delta=reward_delta,
    description='replacement cost reduced by 0.5',
)
print('Manager summary: reducing replacement cost by 0.5 changes the')
print('replacement probability by at most', float(np.max(np.abs(counterfactual.policy_change))))
print('and changes mean model value by', counterfactual.welfare_change)
assert np.max(np.abs(counterfactual.policy_change)) > 0

restored = pickle.loads(pickle.dumps(model))
np.testing.assert_allclose(restored.predict_proba(np.arange(n_states)), model.policy_)
np.testing.assert_allclose(restored.reward_, model.reward_)
print('serialization: exact supported-output parity')

## Interpretation boundary

The `se_` and `pvalues_` fields attached to a feature projection are descriptive regression diagnostics, not sampling uncertainty. Use `compute_se=True` for whole-trajectory bootstrap intervals. The Table 2 replication separately labels the authors' simulation-only best-true-MAPE epoch selection; deployable fits never use that oracle.